# UVIF-Q Fast Upgrade Notebook

This notebook is a fast companion to the full UVIF-Q prototype validation run. The full gradient-based notebook remains the main proof-of-concept. This upgraded version is designed to add reviewer-critical evidence with much lower runtime:

- repeated runs over three random seeds,
- a reduced set of UVIF-Q configurations,
- confidence intervals,
- focused adversarial/noise robustness evaluation,
- calibration and reliability outputs,
- and a regenerated `Framework_Overview.png`.

The notebook intentionally avoids claiming quantum advantage. It validates the operational role of UVIF-Q as a multi-objective reliability layer for hybrid quantum-classical learning under NISQ-inspired noise.

In [ ]:
# ============================================================
# Cell 1 — Environment setup and Google Drive paths
# ============================================================

import os, sys, time, math, json, warnings, subprocess
from pathlib import Path
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Outputs/UVIF_Q_Fast_Upgrade')
else:
    BASE_DIR = Path.cwd() / 'UVIF_Q_Fast_Upgrade'

FIG_DIR = BASE_DIR / 'figures'
TABLE_DIR = BASE_DIR / 'tables'
OUTPUT_DIR = BASE_DIR / 'outputs'
for d in [BASE_DIR, FIG_DIR, TABLE_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = OUTPUT_DIR / 'outputs_summary_fast_upgrade.txt'

def log(msg):
    stamp = time.strftime('%H:%M:%S')
    line = f'[{stamp}] {msg}'
    print(line)
    with open(SUMMARY_PATH, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    f.write('UVIF-Q Fast Upgrade — Outputs Summary\n' + '='*70 + '\n')

log(f'BASE_DIR: {BASE_DIR}')
log(f'Python: {sys.version.split()[0]}')

In [ ]:
# ============================================================
# Cell 2 — Install/import dependencies
# ============================================================

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import pennylane as qml
except Exception:
    log('Installing pennylane ...')
    pip_install('pennylane')
    import pennylane as qml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, brier_score_loss, log_loss)

log('Dependencies imported successfully.')

In [ ]:
# ============================================================
# Cell 3 — Runtime configuration
# ============================================================

RANDOM_SEEDS = [11, 22, 33]        # reviewer-safe repeated runs without excessive runtime
N_CANDIDATE_CIRCUITS = 6           # lightweight variational candidate search
N_QUBITS = 4
N_LAYERS = 1
NOISE_TRAIN_LEVELS = [0.00, 0.04]  # used by decoherence-aware/full configurations
NOISE_SWEEP_LEVELS = [0.00, 0.02, 0.05, 0.08, 0.12]
ADV_EPS = 0.18

CONFIGS = {
    'TaskOnly_QML':      {'alpha':0.00, 'beta':0.00, 'gamma':0.00, 'delta':0.00, 'epsilon':0.00},
    'Uncertainty_U':     {'alpha':0.10, 'beta':0.00, 'gamma':0.00, 'delta':0.00, 'epsilon':0.00},
    'Risk_R':            {'alpha':0.00, 'beta':0.22, 'gamma':0.00, 'delta':0.00, 'epsilon':0.00},
    'Decoherence_DQ':    {'alpha':0.00, 'beta':0.00, 'gamma':0.00, 'delta':0.00, 'epsilon':0.45},
    'Full_UVIF_Q':       {'alpha':0.08, 'beta':0.16, 'gamma':0.02, 'delta':0.05, 'epsilon':0.35},
}

log(f'Seeds: {RANDOM_SEEDS}')
log(f'Configurations: {list(CONFIGS)}')
log('Fast upgrade mode: reduced configurations, candidate quantum circuits, repeated seeds, focused sweeps.')

In [ ]:
# ============================================================
# Cell 4 — Utility functions
# ============================================================

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

def expected_calibration_error(y_true, prob, n_bins=10):
    prob = np.asarray(prob)
    y_true = np.asarray(y_true)
    conf = np.maximum(prob, 1 - prob)
    pred = (prob >= 0.5).astype(int)
    ece = 0.0
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any():
            acc = np.mean(pred[mask] == y_true[mask])
            c = np.mean(conf[mask])
            ece += np.mean(mask) * abs(acc - c)
    return float(ece)

def temperature_scale_probs(prob, y_val, grid=None):
    if grid is None:
        grid = np.linspace(0.6, 2.6, 21)
    p = np.clip(prob, 1e-6, 1-1e-6)
    logits = np.log(p / (1 - p))
    best_T, best_loss = 1.0, 1e9
    for T in grid:
        scaled = sigmoid(logits / T)
        loss = log_loss(y_val, np.vstack([1-scaled, scaled]).T, labels=[0,1])
        if loss < best_loss:
            best_loss, best_T = loss, T
    return best_T

def apply_temperature(prob, T):
    p = np.clip(prob, 1e-6, 1-1e-6)
    logits = np.log(p / (1 - p))
    return sigmoid(logits / T)

def metric_row(y_true, prob, name, seed, extra=None):
    pred = (prob >= 0.5).astype(int)
    row = {
        'seed': seed,
        'model': name,
        'accuracy': accuracy_score(y_true, pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, pred),
        'f1': f1_score(y_true, pred),
        'roc_auc': roc_auc_score(y_true, prob),
        'ece': expected_calibration_error(y_true, prob),
        'brier': brier_score_loss(y_true, prob),
    }
    if extra:
        row.update(extra)
    return row

def bootstrap_ci(values, n_boot=2000, seed=123):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(values), len(values))
        boots.append(values[idx].mean())
    return float(np.mean(values)), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


In [ ]:
# ============================================================
# Cell 5 — Dataset preparation
# ============================================================

def prepare_dataset(seed):
    data = load_breast_cancer()
    X = data.data
    # sklearn label: 0 malignant, 1 benign. We flip so positive = malignant risk.
    y = 1 - data.target

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=seed, stratify=y_train_full
    )

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    pca = PCA(n_components=N_QUBITS, random_state=seed).fit(X_train_s)
    X_train_p = pca.transform(X_train_s)
    X_val_p = pca.transform(X_val_s)
    X_test_p = pca.transform(X_test_s)

    # Angle encoding is more stable when bounded.
    clip = 2.5
    X_train_p = np.clip(X_train_p, -clip, clip) / clip * np.pi
    X_val_p = np.clip(X_val_p, -clip, clip) / clip * np.pi
    X_test_p = np.clip(X_test_p, -clip, clip) / clip * np.pi

    return X_train_p, X_val_p, X_test_p, y_train, y_val, y_test, pca.explained_variance_ratio_.sum()

Xtr, Xva, Xte, ytr, yva, yte, ev = prepare_dataset(RANDOM_SEEDS[0])
log(f'Dataset ready. Example split: train={Xtr.shape}, val={Xva.shape}, test={Xte.shape}, PCA explained variance={ev:.3f}')

In [ ]:
# ============================================================
# Cell 6 — Quantum feature extractor
# ============================================================

# The fast notebook uses candidate variational circuits rather than full parameter-shift
# training. This keeps repeated runs feasible while preserving a hybrid QML feature layer.

def make_qnode(noise_p=0.0):
    dev = qml.device('default.mixed', wires=N_QUBITS)

    @qml.qnode(dev, interface='numpy')
    def circuit(x, theta):
        for w in range(N_QUBITS):
            qml.RY(float(x[w]), wires=w)
            qml.RZ(float(0.5 * x[w]), wires=w)
        idx = 0
        for _ in range(N_LAYERS):
            for w in range(N_QUBITS):
                qml.RY(theta[idx], wires=w); idx += 1
                qml.RZ(theta[idx], wires=w); idx += 1
            for w in range(N_QUBITS):
                qml.CNOT(wires=[w, (w+1) % N_QUBITS])
            if noise_p > 0:
                for w in range(N_QUBITS):
                    qml.DepolarizingChannel(noise_p, wires=w)
        return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]
    return circuit

_QNODE_CACHE = {}
def quantum_features(X, theta, noise_p=0.0):
    key = float(noise_p)
    if key not in _QNODE_CACHE:
        _QNODE_CACHE[key] = make_qnode(noise_p=key)
    qnode = _QNODE_CACHE[key]
    feats = np.array([qnode(x, theta) for x in X], dtype=float)
    return feats

def candidate_thetas(seed, n=N_CANDIDATE_CIRCUITS):
    rng = np.random.default_rng(seed)
    n_params = N_LAYERS * N_QUBITS * 2
    return [rng.normal(0.0, 0.35, size=n_params) for _ in range(n)]

_tmp_theta = candidate_thetas(0, 1)[0]
log(f'QNode sanity output: {quantum_features(Xtr[:1], _tmp_theta, noise_p=0.02)[0]}')

In [ ]:
# ============================================================
# Cell 7 — Configuration-specific training/evaluation logic
# ============================================================

def build_features(X, theta, config_name, noise_p=0.0):
    qf = quantum_features(X, theta, noise_p=noise_p)
    # Raw PCA features + quantum measurements. This avoids a purely black-box quantum layer.
    return np.hstack([X, qf])

def train_predict_for_config(X_train, y_train, X_val, y_val, X_test, theta, config_name, seed):
    rng = np.random.default_rng(seed)

    # Configuration-specific data augmentation / noise exposure.
    if config_name in ['Risk_R', 'Full_UVIF_Q']:
        X_aug = np.vstack([X_train, X_train + rng.normal(0, ADV_EPS, X_train.shape)])
        y_aug = np.concatenate([y_train, y_train])
    else:
        X_aug, y_aug = X_train, y_train

    if config_name in ['Decoherence_DQ', 'Full_UVIF_Q']:
        feats_aug = []
        labels_aug = []
        for p in NOISE_TRAIN_LEVELS:
            feats_aug.append(build_features(X_aug, theta, config_name, noise_p=p))
            labels_aug.append(y_aug)
        F_train = np.vstack(feats_aug)
        yy_train = np.concatenate(labels_aug)
    else:
        F_train = build_features(X_aug, theta, config_name, noise_p=0.0)
        yy_train = y_aug

    F_val = build_features(X_val, theta, config_name, noise_p=0.0)
    F_test = build_features(X_test, theta, config_name, noise_p=0.0)

    Cval = 0.7 if config_name in ['Complexity_C'] else 1.0
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', C=Cval, random_state=seed)
    clf.fit(F_train, yy_train)

    val_prob = clf.predict_proba(F_val)[:, 1]
    test_prob = clf.predict_proba(F_test)[:, 1]

    # Uncertainty and full UVIF-Q use lightweight temperature scaling on validation data.
    T = 1.0
    if config_name in ['Uncertainty_U', 'Full_UVIF_Q']:
        T = temperature_scale_probs(val_prob, y_val)
        val_prob = apply_temperature(val_prob, T)
        test_prob = apply_temperature(test_prob, T)

    # Adversarial-like evaluation: perturb test PCA inputs and recompute quantum features.
    X_adv = X_test + rng.normal(0, ADV_EPS, X_test.shape)
    F_adv = build_features(X_adv, theta, config_name, noise_p=0.0)
    adv_prob = clf.predict_proba(F_adv)[:, 1]
    if config_name in ['Uncertainty_U', 'Full_UVIF_Q']:
        adv_prob = apply_temperature(adv_prob, T)

    # Decoherence proxy: average probability drift between clean and noisy quantum features.
    F_noisy = build_features(X_test, theta, config_name, noise_p=0.08)
    noisy_prob = clf.predict_proba(F_noisy)[:, 1]
    if config_name in ['Uncertainty_U', 'Full_UVIF_Q']:
        noisy_prob = apply_temperature(noisy_prob, T)
    dq_proxy = float(np.mean(np.abs(test_prob - noisy_prob)))

    return clf, T, val_prob, test_prob, adv_prob, dq_proxy

def select_candidate_and_evaluate(seed, config_name, coeffs):
    X_train, X_val, X_test, y_train, y_val, y_test, ev = prepare_dataset(seed)
    best = None
    for j, theta in enumerate(candidate_thetas(seed + 1000, N_CANDIDATE_CIRCUITS)):
        clf, T, val_prob, test_prob, adv_prob, dq_proxy = train_predict_for_config(
            X_train, y_train, X_val, y_val, X_test, theta, config_name, seed + j
        )
        val_pred = (val_prob >= 0.5).astype(int)
        val_bacc = balanced_accuracy_score(y_val, val_pred)
        val_ece = expected_calibration_error(y_val, val_prob)
        # Lightweight objective used only for selecting among candidate circuits.
        complexity = N_LAYERS * N_QUBITS * 2 / 20.0
        score = (val_bacc
                 - coeffs['alpha'] * val_ece
                 - coeffs['gamma'] * complexity
                 - coeffs['epsilon'] * dq_proxy)
        if best is None or score > best['score']:
            best = dict(score=score, theta=theta, clf=clf, T=T, test_prob=test_prob,
                        adv_prob=adv_prob, dq_proxy=dq_proxy, candidate=j, ev=ev)

    row = metric_row(y_test, best['test_prob'], config_name, seed, extra={
        'candidate': best['candidate'],
        'selection_score': best['score'],
        'temperature': best['T'],
        'decoherence_proxy': best['dq_proxy'],
        'adversarial_balanced_accuracy': balanced_accuracy_score(y_test, (best['adv_prob'] >= 0.5).astype(int)),
        'pca_explained_variance': best['ev'],
    })
    return row, best, (X_test, y_test)

log('Training/evaluation utilities ready.')

In [ ]:
# ============================================================
# Cell 8 — Run fast repeated-seed experiments
# ============================================================

start = time.time()
all_rows = []
best_objects = {}
test_sets = {}

for seed in RANDOM_SEEDS:
    log(f'--- Seed {seed} ---')
    for config_name, coeffs in CONFIGS.items():
        log(f'Training/evaluating {config_name} | coeffs={coeffs}')
        row, best, test_set = select_candidate_and_evaluate(seed, config_name, coeffs)
        all_rows.append(row)
        best_objects[(seed, config_name)] = best
        test_sets[seed] = test_set
        log(f"{config_name} seed={seed}: bacc={row['balanced_accuracy']:.4f}, ece={row['ece']:.4f}, "
            f"adv_bacc={row['adversarial_balanced_accuracy']:.4f}, dq={row['decoherence_proxy']:.5f}")

results = pd.DataFrame(all_rows)
results_path = TABLE_DIR / 'table_uvifq_fast_repeated_results.csv'
results.to_csv(results_path, index=False)
log(f'Saved repeated-run table: {results_path}')
log(f'Fast repeated experiment completed in {(time.time()-start)/60:.2f} minutes.')
results

In [ ]:
# ============================================================
# Cell 9 — Aggregate statistics with confidence intervals
# ============================================================

summary_rows = []
for model, grp in results.groupby('model'):
    row = {'model': model, 'n_runs': len(grp)}
    for metric in ['accuracy','balanced_accuracy','f1','roc_auc','ece','brier','adversarial_balanced_accuracy','decoherence_proxy']:
        mean, lo, hi = bootstrap_ci(grp[metric].values, seed=42)
        row[f'{metric}_mean'] = mean
        row[f'{metric}_ci95_low'] = lo
        row[f'{metric}_ci95_high'] = hi
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).sort_values('balanced_accuracy_mean', ascending=False)
summary_path = TABLE_DIR / 'table_uvifq_fast_summary_ci.csv'
summary.to_csv(summary_path, index=False)
log(f'Saved CI summary table: {summary_path}')
summary

In [ ]:
# ============================================================
# Cell 10 — Focused noise sweep
# ============================================================

sweep_rows = []
focus_models = ['TaskOnly_QML', 'Decoherence_DQ', 'Full_UVIF_Q']

for seed in RANDOM_SEEDS:
    X_test, y_test = test_sets[seed]
    for model in focus_models:
        obj = best_objects[(seed, model)]
        clf, theta, T = obj['clf'], obj['theta'], obj['T']
        for p in NOISE_SWEEP_LEVELS:
            F_noise = build_features(X_test, theta, model, noise_p=p)
            prob = clf.predict_proba(F_noise)[:, 1]
            if model in ['Uncertainty_U', 'Full_UVIF_Q']:
                prob = apply_temperature(prob, T)
            pred = (prob >= 0.5).astype(int)
            sweep_rows.append({
                'seed': seed, 'model': model, 'noise_p': p,
                'balanced_accuracy': balanced_accuracy_score(y_test, pred),
                'ece': expected_calibration_error(y_test, prob),
                'brier': brier_score_loss(y_test, prob),
            })

noise_sweep = pd.DataFrame(sweep_rows)
noise_path = TABLE_DIR / 'table_uvifq_noise_sweep.csv'
noise_sweep.to_csv(noise_path, index=False)
log(f'Saved noise sweep table: {noise_path}')
noise_sweep.head()

In [ ]:
# ============================================================
# Cell 11 — Generate publication-ready figures
# ============================================================

# 1. Ablation performance with repeated-run means
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = summary.sort_values('balanced_accuracy_mean', ascending=True)
y = np.arange(len(plot_df))
means = plot_df['balanced_accuracy_mean'].values
lo = plot_df['balanced_accuracy_ci95_low'].values
hi = plot_df['balanced_accuracy_ci95_high'].values
ax.barh(y, means)
ax.errorbar(means, y, xerr=[means-lo, hi-means], fmt='none', capsize=3)
ax.set_yticks(y)
ax.set_yticklabels(plot_df['model'])
ax.set_xlabel('Balanced Accuracy (mean ± 95% bootstrap CI)')
ax.set_title('UVIF-Q Fast Repeated-Run Ablation Performance')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
path = FIG_DIR / 'Fig_UVIFQ_Fast_Ablation_Performance_CI.png'
fig.savefig(path, dpi=300, bbox_inches='tight')
plt.show(); log(f'Saved {path}')

# 2. Calibration ECE
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = summary.sort_values('ece_mean', ascending=False)
y = np.arange(len(plot_df))
means = plot_df['ece_mean'].values
lo = plot_df['ece_ci95_low'].values
hi = plot_df['ece_ci95_high'].values
ax.barh(y, means)
ax.errorbar(means, y, xerr=[means-lo, hi-means], fmt='none', capsize=3)
ax.set_yticks(y)
ax.set_yticklabels(plot_df['model'])
ax.set_xlabel('Expected Calibration Error (lower is better)')
ax.set_title('UVIF-Q Calibration Behavior Across Repeated Runs')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
path = FIG_DIR / 'Fig_UVIFQ_Fast_Calibration_ECE_CI.png'
fig.savefig(path, dpi=300, bbox_inches='tight')
plt.show(); log(f'Saved {path}')

# 3. Noise sweep
fig, ax = plt.subplots(figsize=(8, 5))
for model, grp in noise_sweep.groupby('model'):
    agg = grp.groupby('noise_p')['balanced_accuracy'].agg(['mean','std']).reset_index()
    ax.plot(agg['noise_p'], agg['mean'], marker='o', label=model)
    ax.fill_between(agg['noise_p'], agg['mean']-agg['std'], agg['mean']+agg['std'], alpha=0.15)
ax.set_xlabel('Depolarizing noise probability')
ax.set_ylabel('Balanced Accuracy')
ax.set_title('Focused UVIF-Q Noise Sensitivity Sweep')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
path = FIG_DIR / 'Fig_UVIFQ_Fast_Noise_Sweep.png'
fig.savefig(path, dpi=300, bbox_inches='tight')
plt.show(); log(f'Saved {path}')

# 4. Robustness vs calibration map
fig, ax = plt.subplots(figsize=(7, 5))
for _, r in summary.iterrows():
    ax.scatter(r['ece_mean'], r['adversarial_balanced_accuracy_mean'], s=80)
    ax.text(r['ece_mean'], r['adversarial_balanced_accuracy_mean'], ' ' + r['model'], fontsize=8, va='center')
ax.set_xlabel('ECE (lower is better)')
ax.set_ylabel('Adversarial Balanced Accuracy')
ax.set_title('UVIF-Q Reliability–Robustness Trade-off')
ax.grid(alpha=0.25)
fig.tight_layout()
path = FIG_DIR / 'Fig_UVIFQ_Fast_Reliability_Robustness_Map.png'
fig.savefig(path, dpi=300, bbox_inches='tight')
plt.show(); log(f'Saved {path}')

In [ ]:
# ============================================================
# Cell 12 — Framework overview figure
# ============================================================

fig, ax = plt.subplots(figsize=(13, 5))
ax.axis('off')

boxes = [
    ('Classical data\n+ preprocessing', 0.06),
    ('Quantum feature\nencoding', 0.23),
    ('Variational circuit\nmeasurement', 0.40),
    ('NISQ noise +\ndecoherence proxy', 0.57),
    ('Classical decision\nlayer', 0.74),
    ('UVIF-Q equilibrium\nobjective', 0.90),
]

for text, x in boxes:
    ax.text(x, 0.56, text, ha='center', va='center', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.45', fc='white', ec='black', lw=1.2),
            transform=ax.transAxes)

for i in range(len(boxes)-1):
    x0 = boxes[i][1] + 0.07
    x1 = boxes[i+1][1] - 0.07
    ax.annotate('', xy=(x1, 0.56), xytext=(x0, 0.56), xycoords=ax.transAxes,
                arrowprops=dict(arrowstyle='->', lw=1.2))

ax.text(0.50, 0.16,
        r'$J_{UVIF-Q}=L_{task}+\alpha U+\beta R+\gamma C-\delta I+\epsilon D_Q$',
        ha='center', va='center', fontsize=13, transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='black', lw=1.0))
ax.annotate('', xy=(0.90, 0.43), xytext=(0.58, 0.22), xycoords=ax.transAxes,
            arrowprops=dict(arrowstyle='->', lw=1.2, linestyle='--'))
ax.set_title('Framework Overview: Fast UVIF-Q Hybrid Quantum-Classical Validation Pipeline', fontsize=14, pad=16)
fig.tight_layout()
path = FIG_DIR / 'Framework_Overview.png'
fig.savefig(path, dpi=300, bbox_inches='tight')
plt.show(); log(f'Saved framework figure: {path}')

In [ ]:
# ============================================================
# Cell 13 — Manuscript-ready interpretation block
# ============================================================

# Extract key numbers for direct manuscript use.
base = summary[summary['model'] == 'TaskOnly_QML'].iloc[0]
full = summary[summary['model'] == 'Full_UVIF_Q'].iloc[0]

interpretation = f'''
Key quantitative findings from the fast upgraded repeated-run notebook
----------------------------------------------------------------------
Dataset: Breast cancer diagnostic benchmark with four PCA components used for quantum angle encoding.
Seeds: {RANDOM_SEEDS}
Configurations: {list(CONFIGS.keys())}

TaskOnly_QML balanced accuracy: {base['balanced_accuracy_mean']:.4f} [{base['balanced_accuracy_ci95_low']:.4f}, {base['balanced_accuracy_ci95_high']:.4f}]
Full_UVIF_Q balanced accuracy: {full['balanced_accuracy_mean']:.4f} [{full['balanced_accuracy_ci95_low']:.4f}, {full['balanced_accuracy_ci95_high']:.4f}]

TaskOnly_QML ECE: {base['ece_mean']:.4f} [{base['ece_ci95_low']:.4f}, {base['ece_ci95_high']:.4f}]
Full_UVIF_Q ECE: {full['ece_mean']:.4f} [{full['ece_ci95_low']:.4f}, {full['ece_ci95_high']:.4f}]

TaskOnly_QML adversarial balanced accuracy: {base['adversarial_balanced_accuracy_mean']:.4f}
Full_UVIF_Q adversarial balanced accuracy: {full['adversarial_balanced_accuracy_mean']:.4f}

Interpretation:
This fast companion experiment does not claim quantum advantage. It tests whether the UVIF-Q multi-objective layer
produces measurable reliability, robustness, calibration, and noise-sensitivity behavior across repeated seeds. The full
gradient-based notebook remains the primary prototype; this notebook provides faster repeated-run evidence and focused
noise-sweep diagnostics for reviewer defensibility.
'''

print(interpretation)
with open(OUTPUT_DIR / 'manuscript_interpretation_fast_upgrade.txt', 'w', encoding='utf-8') as f:
    f.write(interpretation)
log(f'Saved manuscript interpretation: {OUTPUT_DIR / "manuscript_interpretation_fast_upgrade.txt"}')

In [ ]:
# ============================================================
# Cell 14 — File index
# ============================================================

files = []
for folder in [FIG_DIR, TABLE_DIR, OUTPUT_DIR]:
    for p in sorted(folder.glob('*')):
        files.append({'folder': str(folder), 'filename': p.name, 'path': str(p)})
file_index = pd.DataFrame(files)
file_index_path = OUTPUT_DIR / 'file_index.csv'
file_index.to_csv(file_index_path, index=False)
log(f'Saved file index: {file_index_path}')
file_index